# Real dataset preparation: Cdataset for drug repositioning

## Highlights:

- Cdataset is widely used in drug repositioning dataset and it is treated as one benchmark dataset. As compareable to existing methods, we used the same one from DRIMC, where the Cdataset includes 2353 known drug-disease associations involving **658 drugs** (as columns) and **409 diseases** (as rows).

- The data sources for drugs side information include drug chemical structure ($S_d^1$), Pfam domain annotation of drug targets ($S_d^2$) and gene ontology term of targets ($S_d^3$). 

- The data sources for disease include phenotype information ($S_t^1$) form OMIM database. 

## Remarks

- Cdataset is an expansion of Fdataset, where Fdataset is also a popular dataset with drug–disease associations compiled from DrugBank and OMIM. [Link](https://zenodo.org/records/8357512) is summarized by Berry et. al., showing different studies that used those datasets. 

## Workflow



---

## 1. Load in the dataset

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
PATH_TO_EXP = (
    "/Users/sijianfan/Documents/projects/BiSSGL/datasets/realAnalysis/cdataset"
)

In [373]:
# -------------------------
# Feature config (tune here)
# -------------------------
# Goal: slightly reduce feature dimensionality while keeping coverage/quality.

# Drug features
ECFP_N_BITS = 1024  # can be 512
ECFP_RADIUS = 2

PFAM_MIN_FREQ = 10  # keep domains that appear in >= this many drugs
PFAM_MAX_FRAC = 0.75  # drop domains that appear in > PFAM_MAX_FRAC * n_drugs
PFAM_TOP_K = 100  # cap number of Pfam features (None = no cap)

GO_MIN_FREQ = 25
GO_MAX_FRAC = 0.75
GO_TOP_K = {"BP": 1000, "MF": 500, "CC": 200}  # caps per GO aspect

# Disease features
HPO_MIN_FREQ = 5
HPO_MAX_FRAC = 0.75
HPO_TOP_K = 1600  # cap number of HPO features

# (Optional) If your disease-gene features are extremely sparse / low-coverage,
# consider disabling them in the final V matrix:
USE_DISEASE_GENE_FEATURES = True

In [4]:
df_Y = pd.read_table(os.path.join(PATH_TO_EXP, "c_admat_dgc.txt"), index_col=0)
print(f"The dataset has diseases and drugs: {df_Y.shape}")

The dataset has diseases and drugs: (409, 658)


---

## 2. Drug side information

### 1. Parse DrugBank XML

In [ ]:
import xml.etree.ElementTree as ET
from collections import defaultdict


def _extract_uniprot_id(polypeptide, ns):
    """Best-effort extraction of UniProt accession from a DrugBank <polypeptide>."""
    if polypeptide is None:
        return None

    # 1) DrugBank often stores UniProt accession in the 'id' attribute (sometimes with source='Swiss-Prot'/'TrEMBL').
    pid = polypeptide.attrib.get("id")
    src = (polypeptide.attrib.get("source") or "").lower()
    if pid and ("uniprot" in src or "swiss" in src or "trembl" in src):
        return pid

    # 2) Fallback: search external-identifiers
    for ext in polypeptide.findall(
        "db:external-identifiers/db:external-identifier", ns
    ):
        resource = (
            ext.findtext("db:resource", default="", namespaces=ns) or ""
        ).lower()
        identifier = ext.findtext("db:identifier", default=None, namespaces=ns)
        if identifier and "uniprot" in resource:
            return identifier

    # 3) Final fallback: sometimes attribute id is already the accession even if source missing
    return pid


def parse_drugbank(xml_file, drugbank_ids):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    ns = {"db": "http://www.drugbank.ca"}

    drug_smiles = {}
    drug_targets = defaultdict(set)

    # NOTE: iterparse is faster for huge XMLs, but ElementTree.parse is OK for moderate sizes.
    for drug in root.findall("db:drug", ns):
        db_id_elem = drug.find("db:drugbank-id[@primary='true']", ns)
        if db_id_elem is None or db_id_elem.text is None:
            continue
        db_id = db_id_elem.text.strip()

        if db_id not in drugbank_ids:
            continue

        # ---- SMILES (best-effort) ----
        smiles = None
        for prop in drug.findall(".//db:property", ns):
            kind = prop.find("db:kind", ns)
            if kind is not None and kind.text == "SMILES":
                val = prop.find("db:value", ns)
                if val is not None and val.text:
                    smiles = val.text.strip()
                break
        drug_smiles[db_id] = smiles

        # ---- Targets: store UniProt accessions if possible ----
        for target in drug.findall("db:targets/db:target", ns):
            polypeptide = target.find("db:polypeptide", ns)
            uniprot = _extract_uniprot_id(polypeptide, ns)
            if uniprot:
                drug_targets[db_id].add(uniprot)

    return drug_smiles, drug_targets

In [6]:
PATH_TO_XML = os.path.join(PATH_TO_EXP, "full database.xml")

In [7]:
drugbank_ids = df_Y.columns
drug_smiles, drug_targets = parse_drugbank(PATH_TO_XML, drugbank_ids)

### 2. Chemical structure → ECFP fingerprints

Updated ECFP code (future-proof): Switch to MorganGenerator

In [109]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import numpy as np


def build_ecfp(drug_smiles, drug_index, n_bits=1024, radius=2):
    n_drugs = len(drug_index)
    X = np.zeros((n_drugs, n_bits), dtype=np.int8)

    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

    for db_id, idx in drug_index.items():
        smiles = drug_smiles.get(db_id)
        if not smiles:
            continue

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue

        fp = generator.GetFingerprint(mol)
        X[idx, :] = np.array(fp)

    return X

In [110]:
drugbank_ids = df_Y.columns
drug_index = {db_id: i for i, db_id in enumerate(drugbank_ids)}

U_ecfp = build_ecfp(drug_smiles, drug_index, n_bits=ECFP_N_BITS, radius=ECFP_RADIUS)

In [111]:
U_ecfp

array([[0, 1, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int8)

In [113]:
print(U_ecfp.shape)  # should be (n_drugs, 512 or 1024 based on your choice)
print(U_ecfp.sum(axis=1)[:10])  # fingerprint density

(658, 1024)
[115  94  71   0 126  12  25  25  46  56]


### 3. Pfam domain features
#### 3.1 Load UniProt → Pfam mapping

In [114]:
def load_uniprot_pfam(pfam_file):
    uniprot2pfam = defaultdict(set)
    with open(pfam_file) as f:
        for line in f:
            if line.startswith("#"):
                continue
            fields = line.strip().split("\t")
            uniprot, pfam = fields[0], fields[4]
            uniprot2pfam[uniprot].add(pfam)
    return uniprot2pfam

In [15]:
pfam_file = os.path.join(PATH_TO_EXP, "Pfam-A.regions.tsv")
uniprot2pfam = load_uniprot_pfam(pfam_file)

### 3.2 Build drug–Pfam matrix

In [166]:
from collections import Counter, defaultdict
import numpy as np


def build_pfam_matrix(
    drug_targets, uniprot2pfam, drug_index, min_freq=2, max_frac=0.8, top_k=None
):
    """Build a drug × Pfam binary matrix.

    Key fixes vs naive versions:
      - Count Pfam frequency per drug (not per protein occurrence) for filtering.
      - Optionally drop overly-common domains (max_frac) and cap dimensionality (top_k).
    """

    # Collect unique Pfams per drug
    drug_pfam = {}
    for drug, prots in drug_targets.items():
        pfams = set()
        for p in prots:
            pfams.update(uniprot2pfam.get(p, set()))
        if pfams:
            drug_pfam[drug] = pfams

    # Count in how many drugs each Pfam appears
    pfam_counts = Counter()
    for pfams in drug_pfam.values():
        pfam_counts.update(pfams)

    N = len(drug_index)
    max_count = int(max_frac * N)

    kept = [p for p, c in pfam_counts.items() if c >= min_freq and c <= max_count]

    # Sort by frequency (desc) then id to make results deterministic
    kept = sorted(kept, key=lambda p: (-pfam_counts[p], p))

    if top_k is not None:
        kept = kept[: int(top_k)]

    pfam_index = {p: i for i, p in enumerate(kept)}

    X = np.zeros((N, len(kept)), dtype=np.int8)

    for drug, pfams in drug_pfam.items():
        if drug not in drug_index:
            continue
        i = drug_index[drug]
        for p in pfams:
            j = pfam_index.get(p)
            if j is not None:
                X[i, j] = 1

    return X, pfam_index, pfam_counts

In [256]:
U_pfam, pfam_index, pfam_counts = build_pfam_matrix(
    drug_targets=drug_targets,
    uniprot2pfam=uniprot2pfam,
    drug_index=drug_index,
    min_freq=PFAM_MIN_FREQ,
    max_frac=PFAM_MAX_FRAC,
    top_k=PFAM_TOP_K,
)

In [257]:
print(U_pfam.shape)
print("Avg Pfam domains per drug:", U_pfam.sum(axis=1).mean())
print("Drugs with no Pfam domains:", (U_pfam.sum(axis=1) == 0).sum())
print("Most common Pfam domain count:", U_pfam.sum(axis=0).max())

(658, 65)
Avg Pfam domains per drug: 2.594224924012158
Drugs with no Pfam domains: 92
Most common Pfam domain count: 260


Check sparsity

In [268]:
def check_sparsity(X, name):
    density = np.count_nonzero(X) / X.size
    avg_per_drug = np.count_nonzero(X, axis=1).mean()
    avg_per_feature = np.count_nonzero(X, axis=0).mean()

    print(f"{name}:")
    print(f"  shape: {X.shape}")
    print(f"  density: {density:.6f}")
    print(f"  avg Pfams per drug: {avg_per_drug:.2f}")
    print(f"  avg drugs per Pfam: {avg_per_feature:.2f}")

In [269]:
check_sparsity(U_pfam, "Pfam")

Pfam:
  shape: (658, 65)
  density: 0.039911
  avg Pfams per drug: 2.59
  avg drugs per Pfam: 26.26


### 4. GO term features
#### 4.1 Load UniProt → GO

In [19]:
def load_uniprot_go(go_file):
    uniprot2go = defaultdict(set)
    with open(go_file) as f:
        for line in f:
            if line.startswith("!"):
                continue
            fields = line.strip().split("\t")
            uniprot = fields[1]
            go_id = fields[4]
            uniprot2go[uniprot].add(go_id)
    return uniprot2go

Split the GO by 3 aspects

In [20]:
from collections import defaultdict


def load_uniprot_go(go_file):
    """
    Returns:
        uniprot2go = {
            'BP': {uniprot: set(GO terms)},
            'MF': {uniprot: set(GO terms)},
            'CC': {uniprot: set(GO terms)},
        }
    """
    uniprot2go = {
        "BP": defaultdict(set),
        "MF": defaultdict(set),
        "CC": defaultdict(set),
    }

    aspect_map = {
        "P": "BP",
        "F": "MF",
        "C": "CC",
    }

    with open(go_file, "r") as f:
        for line in f:
            if line.startswith("!"):
                continue

            fields = line.rstrip("\n").split("\t")

            uniprot = fields[1]
            go_id = fields[4]
            aspect = fields[8]

            if aspect in aspect_map:
                uniprot2go[aspect_map[aspect]][uniprot].add(go_id)

    return uniprot2go

In [21]:
go_file = os.path.join(PATH_TO_EXP, "goa_human.gaf")
uniprot2go = load_uniprot_go(go_file)

In [22]:
for aspect in ["BP", "MF", "CC"]:
    print(aspect, "proteins annotated:", len(uniprot2go[aspect]))

BP proteins annotated: 17791
MF proteins annotated: 18290
CC proteins annotated: 19019


In [23]:
from collections import defaultdict

drug_go = {
    "BP": defaultdict(set),
    "MF": defaultdict(set),
    "CC": defaultdict(set),
}

for drug, prots in drug_targets.items():
    for p in prots:
        for aspect in ["BP", "MF", "CC"]:
            if p in uniprot2go[aspect]:
                drug_go[aspect][drug].update(uniprot2go[aspect][p])

#### 4.2 Build drug–GO matrix

Split by 3 aspects

In [140]:
import numpy as np
from collections import Counter


def build_go_matrix(drug_go_aspect, drug_index, min_freq=5, max_frac=0.8, top_k=None):
    """Build a drug × GO binary matrix for ONE aspect (BP/MF/CC).

    - Frequency is counted per drug (each term counted once per drug).
    - You can drop too-rare terms (min_freq), too-common terms (max_frac), and cap features (top_k).
    """
    go_counts = Counter()
    for gos in drug_go_aspect.values():
        go_counts.update(set(gos))

    N = len(drug_index)
    max_count = int(max_frac * N)

    kept = [go for go, c in go_counts.items() if (c >= min_freq and c <= max_count)]
    kept = sorted(kept, key=lambda go: (-go_counts[go], go))

    if top_k is not None:
        kept = kept[: int(top_k)]

    go_index = {go: j for j, go in enumerate(kept)}

    X = np.zeros((N, len(go_index)), dtype=np.int8)

    for drug, gos in drug_go_aspect.items():
        if drug not in drug_index:
            continue
        i = drug_index[drug]
        for go in gos:
            j = go_index.get(go)
            if j is not None:
                X[i, j] = 1

    return X, go_index, go_counts

In [141]:
U_go_bp, go_bp_index, go_bp_counts = build_go_matrix(
    drug_go["BP"],
    drug_index,
    min_freq=GO_MIN_FREQ,
    max_frac=GO_MAX_FRAC,
    top_k=GO_TOP_K.get("BP"),
)

U_go_mf, go_mf_index, go_mf_counts = build_go_matrix(
    drug_go["MF"],
    drug_index,
    min_freq=GO_MIN_FREQ,
    max_frac=GO_MAX_FRAC,
    top_k=GO_TOP_K.get("MF"),
)

U_go_cc, go_cc_index, go_cc_counts = build_go_matrix(
    drug_go["CC"],
    drug_index,
    min_freq=GO_MIN_FREQ,
    max_frac=GO_MAX_FRAC,
    top_k=GO_TOP_K.get("CC"),
)

In [142]:
print("GO:BP", U_go_bp.shape, "avg terms:", U_go_bp.sum(axis=1).mean())

print("GO:MF", U_go_mf.shape, "avg terms:", U_go_mf.sum(axis=1).mean())

print("GO:CC", U_go_cc.shape, "avg terms:", U_go_cc.sum(axis=1).mean())

GO:BP (658, 505) avg terms: 44.98024316109422
GO:MF (658, 173) avg terms: 17.25987841945289
GO:CC (658, 122) avg terms: 16.788753799392097


Check the sparsity

In [270]:
def check_sparsity(X, name):
    density = np.count_nonzero(X) / X.size
    avg_per_drug = np.count_nonzero(X, axis=1).mean()
    avg_per_feature = np.count_nonzero(X, axis=0).mean()

    print(f"{name}:")
    print(f"  shape: {X.shape}")
    print(f"  density: {density:.6f}")
    print(f"  avg GO terms per drug: {avg_per_drug:.2f}")
    print(f"  avg drugs per GO term: {avg_per_feature:.2f}")

In [271]:
check_sparsity(U_go_bp, "BP")
check_sparsity(U_go_mf, "MF")
check_sparsity(U_go_cc, "CC")

BP:
  shape: (658, 505)
  density: 0.089070
  avg GO terms per drug: 44.98
  avg drugs per GO term: 58.61
MF:
  shape: (658, 173)
  density: 0.099768
  avg GO terms per drug: 17.26
  avg drugs per GO term: 65.65
CC:
  shape: (658, 122)
  density: 0.137613
  avg GO terms per drug: 16.79
  avg drugs per GO term: 90.55


### 5. Save out the U side information

In [259]:
U_blocks = {
    "ecfp": U_ecfp,
    "pfam": U_pfam,
    "go_bp": U_go_bp,
    "go_mf": U_go_mf,
    "go_cc": U_go_cc,
}

In [260]:
U = np.hstack(
    [
        U_blocks["ecfp"],
        U_blocks["pfam"],
        U_blocks["go_bp"],
        U_blocks["go_mf"],
        U_blocks["go_cc"],
    ]
)

In [261]:
print(U.shape)

(658, 1889)


In [262]:
feature_names = []

# ECFP
feature_names += [f"ECFP_{i}" for i in range(U_blocks["ecfp"].shape[1])]

# Pfam
feature_names += [f"PFAM_{pfam}" for pfam in pfam_index]

# GO
feature_names += [f"GO_BP_{go}" for go in go_bp_index]
feature_names += [f"GO_MF_{go}" for go in go_mf_index]
feature_names += [f"GO_CC_{go}" for go in go_cc_index]

In [263]:
# Ensure correct drug order
drug_ids_ordered = [d for d, _ in sorted(drug_index.items(), key=lambda x: x[1])]

df_U = pd.DataFrame(U, index=drug_ids_ordered, columns=feature_names)

In [264]:
df_Y.columns

Index(['DB00014', 'DB00035', 'DB00091', 'DB00104', 'DB00115', 'DB00122',
       'DB00125', 'DB00126', 'DB00131', 'DB00136',
       ...
       'DB08801', 'DB08802', 'DB08804', 'DB08820', 'DB08824', 'DB08835',
       'DB08896', 'DB08901', 'DB08906', 'DB08907'],
      dtype='object', length=658)

In [265]:
df_U.index

Index(['DB00014', 'DB00035', 'DB00091', 'DB00104', 'DB00115', 'DB00122',
       'DB00125', 'DB00126', 'DB00131', 'DB00136',
       ...
       'DB08801', 'DB08802', 'DB08804', 'DB08820', 'DB08824', 'DB08835',
       'DB08896', 'DB08901', 'DB08906', 'DB08907'],
      dtype='object', length=658)

In [266]:
output_path = os.path.join(PATH_TO_EXP, "drugs_features.csv")
df_U.to_csv(output_path)

---

## 3. Diseases side information as features

In [37]:
df_Y.index

Index(['D102100', 'D102300', 'D102400', 'D102500', 'D103100', 'D103230',
       'D103285', 'D103780', 'D104130', 'D104300',
       ...
       'D608232', 'D608266', 'D608320', 'D608437', 'D608456', 'D608583',
       'D608622', 'D608636', 'D608895', 'D608907'],
      dtype='object', length=409)

1.	HPO (Human Phenotype Ontology) binary matrix — best surrogate for MimMiner.
OMIM ↔ HPO mappings exist (Monarch/OMIM annotations). Binary presence of HPO terms per disease; propagate to ancestors to reduce sparsity.
2.	OMIM text TF–IDF → low-dim embedding — captures free-text phenotype descriptions (MimMiner is text-based). Use TF–IDF → truncated SVD (LSA) or PCA → treat components as features.
3.	Disease–gene matrix (you already planned via OMIM) — mechanistic and essential.
4.	Pathway / Reactome / KEGG aggregation of disease genes — lower-dim and often highly predictive.
5.	Similarity-derived embeddings — convert pairwise MimMiner-like similarity into features via MDS / spectral embedding / diffusion maps. Useful if you already compute a pairwise similarity.
6.	Disease ontology / MeSH / DO terms — hierarchical disease categories as features (coarser than HPO).

Combine a subset (1,2,3,4) for best performance & interpretability.

### 1. Parse OMIM JSON

In [38]:
import os

OMIM_CACHE_DIR = os.path.join(PATH_TO_EXP, "./omim_cache")
os.makedirs(OMIM_CACHE_DIR, exist_ok=True)

In [39]:
import requests
import json
import time


def query_omim_entry(omim_id, api_key, cache_dir=OMIM_CACHE_DIR, sleep_time=0.5):
    cache_file = os.path.join(cache_dir, f"{omim_id}.json")

    # use cached version if exists
    if os.path.exists(cache_file):
        with open(cache_file, "r") as f:
            return json.load(f)

    url = "https://api.omim.org/api/entry"
    params = {
        "mimNumber": omim_id,
        "format": "json",
        "apiKey": api_key,
        "include": "geneMap,clinicalSynopsis",
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print(f"OMIM API error for {omim_id}: {response.status_code}")
        return None

    data = response.json()

    with open(cache_file, "w") as f:
        json.dump(data, f, indent=2)

    time.sleep(sleep_time)  # be nice to OMIM servers
    return data

In [40]:
disease_to_omim = {d: [d[1:]] for d in df_Y.index}  # remove leading 'D'

In [41]:
omim_api_key = "STw1icyaR3mDxZYTJWQc5w"

unique_omim_ids = sorted({v[0] for v in disease_to_omim.values()})

omim_raw = {}
for omim_id in unique_omim_ids:
    omim_raw[omim_id] = query_omim_entry(omim_id, omim_api_key)

#### 1.1 Parse OMIM JSON into disease-level dictionaries

In [ ]:
from collections import defaultdict


def parse_omim_raw(disease_to_omim, omim_raw):
    """
    Returns:
        disease_genes: dict {disease_id: set(gene symbols)}
        disease_text: dict {disease_id: string}
    """
    disease_genes = defaultdict(set)
    disease_text = {}

    for disease_id, omim_ids in disease_to_omim.items():
        texts = []

        for omim_id in omim_ids:
            raw = omim_raw.get(omim_id)
            if raw is None:
                continue

            try:
                entry = raw["omim"]["entryList"][0]["entry"]
            except (KeyError, IndexError):
                continue

            # ---- genes ----
            gene_map = entry.get("geneMap")
            if gene_map:
                gene_symbols = gene_map.get("geneSymbols")
                if gene_symbols:
                    # geneSymbols may be:
                    # "BRCA1, BRCA1-AS1"
                    # "CFTR (cystic fibrosis transmembrane conductance regulator)"
                    for g in gene_symbols.replace(";", ",").split(","):
                        g = g.strip()
                        if not g:
                            continue
                        # remove parenthetical content
                        g = g.split("(")[0].strip()
                        disease_genes[disease_id].add(g)

            # ---- text ----
            clin = entry.get("clinicalSynopsis")
            if clin:
                texts.append(" ".join(str(v) for v in clin.values()))

        # merge text from multiple OMIM IDs
        if texts:
            disease_text[disease_id] = " ".join(texts)
        else:
            disease_text[disease_id] = ""

    return disease_genes, disease_text

The below one is not used actually:
- disease_genes: replaced by mim2genes later
- disease_texts: places by hpo later

In [ ]:
disease_genes, disease_texts = parse_omim_raw(
    disease_to_omim=disease_to_omim, omim_raw=omim_raw
)

Check sparsity

In [305]:
def check_disease_gene_stats(disease_genes):
    counts = [len(v) for v in disease_genes.values()]
    print("Num diseases:", len(counts))
    print("Avg genes per disease:", np.mean(counts))
    print("Median genes per disease:", np.median(counts))
    print("Max genes per disease:", np.max(counts))
    print("Min genes per disease:", np.min(counts))

In [306]:
check_disease_gene_stats(disease_genes)

Num diseases: 73
Avg genes per disease: 1.3835616438356164
Median genes per disease: 1.0
Max genes per disease: 5
Min genes per disease: 1


#### 1.2 Map gene symbols → Entrez IDs

In [ ]:
import os, urllib.request
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

MIM2GENE_URL = "https://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen"
MIM2GENE_PATH = os.path.join(PATH_TO_EXP, "mim2gene_medgen.tsv")

if not os.path.exists(MIM2GENE_PATH):
    urllib.request.urlretrieve(MIM2GENE_URL, MIM2GENE_PATH)

mim_df = pd.read_csv(MIM2GENE_PATH, sep="\t", dtype=str)
mim_df.columns = [c.lstrip("#").strip() for c in mim_df.columns]
for c in ["type", "Source", "MedGenCUI", "Comment"]:
    if c in mim_df.columns:
        mim_df[c] = mim_df[c].astype(str).str.strip()


def norm_mim(x):
    """Normalize MIM number to plain digits string or None."""
    if x is None:
        return None
    s = str(x).strip()
    if s in {"", "-", "nan", "None", "NA"}:
        return None
    if s.isdigit():
        return str(int(s))
    for token in ["OMIM:", "MIM:"]:
        if s.startswith(token):
            tail = s[len(token) :].strip()
            if tail.isdigit():
                return str(int(tail))
    return s


mim_df["MIM number"] = mim_df["MIM number"].map(norm_mim)

In [312]:
mim_df

,MIM number,GeneID,type,Source,MedGenCUI,Comment
0,100050,-,phenotype,-,C3149220,-
1,100070,-,phenotype,-,C1853365,-
2,100100,1131,phenotype,GeneMap,C0033770,-
3,100200,-,phenotype,-,C4551519,-
4,100300,57514,phenotype,GeneMap,C4551482,-
...,...,...,...,...,...,...
28569,621494,101954271,gene,-,-,-
28570,621495,5725,phenotype,GeneMap,CN380859,-
28571,621496,51016,gene,-,-,-
28572,621497,51020,gene,-,-,-


In [311]:
# phenotype -> GeneID links
ph = mim_df[(mim_df["type"] == "phenotype") & (mim_df["GeneID"] != "-")].copy()
ph["GeneID"] = ph["GeneID"].astype(int)

# Optional: filter out weaker links using "Comment"
DROP_COMMENTS = set()  # e.g. {"susceptibility", "nondisease", "somatic", "question"}
if DROP_COMMENTS:
    ph = ph[~ph["Comment"].isin(DROP_COMMENTS)]

mim2genes = (
    ph.groupby("MIM number")["GeneID"].apply(lambda s: set(s.tolist())).to_dict()
)

In [322]:
# disease_id -> set(GeneID)
disease_entrez = defaultdict(set)
for disease_id, omim_list in disease_to_omim.items():
    for omim in omim_list:
        m = norm_mim(omim)
        if m and m in mim2genes:
            disease_entrez[disease_id].update(mim2genes[m])

print(
    "Diseases with >=1 gene:",
    sum(len(v) > 0 for v in disease_entrez.values()),
    "/",
    len(disease_entrez),
)
print(
    "Avg genes per disease:", float(np.mean([len(v) for v in disease_entrez.values()]))
)

Diseases with >=1 gene: 224 / 224
Avg genes per disease: 2.3080357142857144


#### 1.3 Build the disease–gene feature matrix V_gene

In [ ]:
from collections import Counter

gene_counts = Counter()
for genes in disease_entrez.values():
    gene_counts.update(genes)

N_diseases = len(df_Y.index)

# frequency filtering (recommended)
GENE_MIN_DISEASES = 1
GENE_MAX_FRAC = 0.8
GENE_TOP_K = None  # e.g. 300

genes_kept = [
    g
    for g, c in gene_counts.items()
    if c >= GENE_MIN_DISEASES and c <= GENE_MAX_FRAC * N_diseases
]
genes_kept = sorted(genes_kept, key=lambda g: (-gene_counts[g], g))
if GENE_TOP_K is not None:
    genes_kept = genes_kept[:GENE_TOP_K]

gene_index = {g: j for j, g in enumerate(sorted(genes_kept))}

V_gene = np.zeros((len(df_Y.index), len(gene_index)), dtype=np.int8)
disease_index = {d: i for i, d in enumerate(df_Y.index)}

for disease, genes in disease_entrez.items():
    if disease not in disease_index:
        continue

    i = disease_index[disease]
    for g in genes:
        if g in gene_index:
            V_gene[i, gene_index[g]] = 1

In [364]:
print(V_gene.shape)
print("Avg genes per disease:", V_gene.sum(axis=1).mean())
print("Diseases with no genes:", (V_gene.sum(axis=1) == 0).sum())

(409, 426)
Avg genes per disease: 1.2640586797066016
Diseases with no genes: 185


### 2. Phenotype features

In [86]:
from collections import defaultdict


def load_omim_hpo(hpoa_file):
    """
    Returns:
        omim2hpo: dict { '154700' : set(HP terms) }
    """
    omim2hpo = defaultdict(set)

    with open(hpoa_file, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")

            disease_id = fields[0]  # e.g. OMIM:154700
            hpo_id = fields[3]  # e.g. HP:0001250

            if disease_id.startswith("OMIM:"):
                omim = disease_id.split(":")[1]
                omim2hpo[omim].add(hpo_id)

    return omim2hpo

In [56]:
hpoa_file = os.path.join(PATH_TO_EXP, "phenotype.hpoa")

omim2hpo = load_omim_hpo(hpoa_file)

In [57]:
disease_hpo = defaultdict(set)

for disease, omim_ids in disease_to_omim.items():
    for omim in omim_ids:
        if omim in omim2hpo:
            disease_hpo[disease].update(omim2hpo[omim])

In [58]:
print(sum(len(v) > 0 for v in disease_hpo.values()), "/", len(disease_index))

352 / 409


In [59]:
from collections import defaultdict


def load_hpo_ancestors(obo_file):
    """
    Returns:
        ancestors: dict { HPO_term -> set(all ancestor HPO terms) }
    """
    parents = defaultdict(set)

    # ---- Step 1: parse parents ----
    current = None
    with open(obo_file, "r") as f:
        for line in f:
            line = line.strip()
            if line == "[Term]":
                current = None
            elif line.startswith("id:"):
                current = line.split("id:")[1].strip()
            elif line.startswith("is_a:") and current:
                parent = line.split("is_a:")[1].split("!")[0].strip()
                parents[current].add(parent)

    # ---- Step 2: compute ancestors with memoization ----
    ancestors = {}

    def get_ancestors(h):
        if h in ancestors:
            return ancestors[h]

        anc = set()
        for p in parents.get(h, []):
            anc.add(p)
            anc.update(get_ancestors(p))

        ancestors[h] = anc
        return anc

    # IMPORTANT: freeze the list of HPO terms
    all_terms = list(parents.keys())

    for h in all_terms:
        get_ancestors(h)

    return ancestors

In [60]:
obo_file = os.path.join(PATH_TO_EXP, "hp.obo")

hpo_ancestors = load_hpo_ancestors(obo_file)

In [61]:
len(hpo_ancestors)
list(hpo_ancestors.items())[:3]

[('HP:0000001', set()),
 ('HP:0000118', {'HP:0000001'}),
 ('HP:0001507', {'HP:0000001', 'HP:0000118'})]

In [62]:
# propagate HPO terms
disease_hpo_expanded = {}

for d, terms in disease_hpo.items():
    expanded = set(terms)
    for t in terms:
        expanded.update(hpo_ancestors.get(t, set()))
    disease_hpo_expanded[d] = expanded

In [374]:
from collections import Counter
import numpy as np

hpo_counts = Counter()
for terms in disease_hpo_expanded.values():
    hpo_counts.update(terms)

N = len(disease_index)
max_count = int(HPO_MAX_FRAC * N)

# keep moderately-informative terms
hpo_terms = [h for h, c in hpo_counts.items() if (c >= HPO_MIN_FREQ and c <= max_count)]

# drop very generic roots if they slip through
DROP_HPO = {"HP:0000001", "HP:0000118"}  # All; Phenotypic abnormality
hpo_terms = [h for h in hpo_terms if h not in DROP_HPO]

# deterministic ordering: by frequency desc, then term id
hpo_terms = sorted(hpo_terms, key=lambda h: (-hpo_counts[h], h))

# optional dimensionality cap
# if HPO_TOP_K is not None:
#     hpo_terms = hpo_terms[: int(HPO_TOP_K)]

hpo_index = {h: j for j, h in enumerate(hpo_terms)}

In [375]:
V_hpo = np.zeros((len(disease_index), len(hpo_index)), dtype=np.int8)

for disease, terms in disease_hpo_expanded.items():
    i = disease_index[disease]
    for h in terms:
        if h in hpo_index:
            V_hpo[i, hpo_index[h]] = 1

Check sparsity

In [376]:
def check_sparsity(X, name):
    density = np.count_nonzero(X) / X.size
    avg_per_drug = np.count_nonzero(X, axis=1).mean()
    avg_per_feature = np.count_nonzero(X, axis=0).mean()

    print(f"{name}:")
    print(f"  shape: {X.shape}")
    print(f"  density: {density:.6f}")
    print(f"  avg HPO per disease: {avg_per_drug:.2f}")
    print(f"  avg diseases per HPO: {avg_per_feature:.2f}")

In [377]:
check_sparsity(V_hpo, "HPO")

HPO:
  shape: (409, 797)
  density: 0.040786
  avg HPO per disease: 32.51
  avg diseases per HPO: 16.68


In [378]:
V_hpo

array([[1, 0, 0, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int8)

Since genes are too sparse with low coverage to the diseases, we only keep HPO.

In [379]:
V_blocks = {
    # "gene": V_gene,
    "hpo": V_hpo,
}

---

### 5. Save out the V side information

In [381]:
USE_DISEASE_GENE_FEATURES = False

In [382]:
import numpy as np

if USE_DISEASE_GENE_FEATURES:
    V = np.hstack([V_gene, V_hpo])  # (n_diseases × (n_gene + n_hpo))
else:
    V = V_hpo  # (n_diseases × n_hpo)

In [383]:
feature_names_V = []

if USE_DISEASE_GENE_FEATURES:
    # gene features
    feature_names_V += [f"GENE_{g}" for g in gene_index]

# HPO features
feature_names_V += [f"HPO_{h}" for h in hpo_index]

assert len(feature_names_V) == V.shape[1]

In [384]:
import pandas as pd

disease_ids_ordered = [d for d, _ in sorted(disease_index.items(), key=lambda x: x[1])]

df_V = pd.DataFrame(V, index=disease_ids_ordered, columns=feature_names_V)

In [385]:
df_V.shape

(409, 797)

In [386]:
output_path = os.path.join(PATH_TO_EXP, "diseases_features.csv")
df_V.to_csv(output_path)

In [387]:
def drop_constant_cols(df, name="X"):
    col_sums = df.sum(axis=0)
    n = df.shape[0]

    all0 = col_sums == 0
    all1 = col_sums == n

    drop_cols = df.columns[all0 | all1].tolist()

    print(
        f"{name}: dropping {len(drop_cols)} constant columns "
        f"({int(all0.sum())} all-0, {int(all1.sum())} all-1)"
    )
    if drop_cols:
        print("  first few dropped:", drop_cols[:10])

    df2 = df.drop(columns=drop_cols)
    print(f"{name}: new shape = {df2.shape}")
    return df2, drop_cols


df_U, dropped_U = drop_constant_cols(df_U, "df_U")
print("Drugs with all-zero rows after drop:", int((df_U.sum(axis=1) == 0).sum()))

df_V, dropped_V = drop_constant_cols(df_V, "df_V")
print("Diseases with all-zero rows after drop:", int((df_V.sum(axis=1) == 0).sum()))

df_U: dropping 1 constant columns (1 all-0, 0 all-1)
  first few dropped: ['ECFP_337']
df_U: new shape = (658, 1888)
Drugs with all-zero rows after drop: 0
df_V: dropping 0 constant columns (0 all-0, 0 all-1)
df_V: new shape = (409, 797)
Diseases with all-zero rows after drop: 57


In [388]:
output_path = os.path.join(PATH_TO_EXP, "drugs_features.csv")
df_U.to_csv(output_path)